# Quantification Over Combined Peak Sets (BigWig-based)

Fast quantification of ATAC-seq signal over **combined peak sets** (unified + species-specific)
produced by `04_cross_species_consensus`.

## Peak Structure
For each species, the combined BED file (`all_peaks_{Species}.bed`) contains:
- **`unified_NNNNNN`** — peaks comparable across genomes (same ID = same peak)
- **`human_peak_NNNNNN`** or **`{species}_peak_NNNNNN`** — species-specific peaks

## Workflow
1. **Load** combined peak sets from `09_combined/` (from notebook 04)
2. **Discover** fragment files per species
3. **Convert** fragments → bigWig (one-time, cached)
4. **Quantify** bigWig signal over all peaks (unified + specific) per species
5. **Save** one feather matrix per species (peaks × samples)

In [ ]:
# Imports
import os
import sys
import glob
import subprocess
import pandas as pd
import numpy as np
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

# Add atac_pipeline to path
sys.path.insert(0, os.path.abspath('..'))

from src.quantification import (
    quantify_bigwig, quantify_bigwig_matrix,
    fragments_to_bigwigs, save_matrix, load_matrix,
)

print("Quantification module loaded (bigWig-based)")
print("   quantify_bigwig()        - single bigWig over peaks")
print("   quantify_bigwig_matrix() - multiple bigWigs -> peaks × samples matrix")
print("   fragments_to_bigwigs()   - convert fragment files → bigWig (cached)")
print("   save/load_matrix         - feather / parquet / tsv I/O")

Quantification module loaded
   quantify()        - single file over peaks
   quantify_matrix() - multiple files -> peaks x samples matrix
   save/load_matrix  - feather / parquet / tsv I/O
   bedtools v2.31.1


## Configuration

In [ ]:
# === CONFIGURATION ===

BASE_PATH = "/cluster/project/treutlein/USERS/jjans"
CROSS_SPECIES_DIR = f"{BASE_PATH}/analysis/adult_intestine/peaks/cross_species_consensus_v2"
GENOMES_REF_DIR = "/cluster/work/treutlein/jjans/data/intestine/nhp_atlas/genomes/reference_"

# Species
SPECIES_LIST = ["Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset", "Human"]

# Chromosome sizes files per species (needed for fragment → bigWig conversion)
CHROM_SIZES = {
    "Human":      f"{BASE_PATH}/analysis/intestine/encode_data/hg38.chrom.sizes",
    "Bonobo":     f"{GENOMES_REF_DIR}/panPan1/fasta/panPan1.chrom.sizes",
    "Chimpanzee": f"{GENOMES_REF_DIR}/panTro3/fasta/panTro3.chrom.sizes",
    "Gorilla":    f"{GENOMES_REF_DIR}/gorGor4/fasta/gorGor4.chrom.sizes",
    "Macaque":    f"{GENOMES_REF_DIR}/Mmul10/fasta/Mmul10.chrom.sizes",
    "Marmoset":   f"{GENOMES_REF_DIR}/calJac1_mito/fasta/calJac1_mito.chrom.sizes",
}

# Output directories
BIGWIG_OUTPUT_DIR = os.path.join(CROSS_SPECIES_DIR, "11_bigwigs")
QUANT_OUTPUT_DIR  = os.path.join(CROSS_SPECIES_DIR, "11_quantification")
os.makedirs(BIGWIG_OUTPUT_DIR, exist_ok=True)
os.makedirs(QUANT_OUTPUT_DIR, exist_ok=True)

# Verify chrom.sizes files exist
print("Chromosome sizes files:")
for sp, path in CHROM_SIZES.items():
    exists = os.path.exists(path)
    print(f"   {sp:<15} {'✅' if exists else '❌'} {path}")

missing = [sp for sp in SPECIES_LIST if sp not in CHROM_SIZES]
if missing:
    print(f"\n⚠️  Missing chrom.sizes for: {missing}")
    print("   These species will be skipped during bigWig conversion.")

print(f"\nBigWig output:  {BIGWIG_OUTPUT_DIR}")
print(f"Quant output:   {QUANT_OUTPUT_DIR}")

## Load Combined Peak Sets

Load `all_peaks_{Species}.bed` from `09_combined/`. Each file contains both
unified peaks (comparable across species) and species-specific peaks.

In [ ]:
# Load combined peak files (unified + species-specific) from 09_combined/
COMBINED_DIR = os.path.join(CROSS_SPECIES_DIR, "09_combined")

SPECIES_PEAK_FILES = {}

print("Loading combined peak files (unified + species-specific):")
print("=" * 70)

for species in SPECIES_LIST:
    peak_file = os.path.join(COMBINED_DIR, f"all_peaks_{species}.bed")
    
    if os.path.exists(peak_file):
        with open(peak_file) as f:
            lines = [l for l in f if l.strip() and not l.startswith('#')]
        n_total = len(lines)
        
        # Count unified vs specific
        n_unified = sum(1 for l in lines if l.split('\t')[3].startswith('unified_'))
        n_specific = n_total - n_unified
        
        SPECIES_PEAK_FILES[species] = peak_file
        print(f"{species:<15} {n_total:>8,} peaks  "
              f"(unified: {n_unified:,}, specific: {n_specific:,})  "
              f"{peak_file}")
    else:
        print(f"{species:<15} NOT FOUND: {peak_file}")

print(f"\nPeak files available for {len(SPECIES_PEAK_FILES)}/{len(SPECIES_LIST)} species")

## Discover Fragment Files

In [ ]:
# Auto-detect fragment files for each species
FRAGMENT_DIR_TEMPLATE = f"{BASE_PATH}/analysis/adult_intestine/peaks/fragment_files/{{species}}"
FRAGMENT_PATTERN = "*.tsv.gz"

SPECIES_FRAGMENTS = {}

print("Scanning for fragment files:")
print("-" * 70)

for species in SPECIES_LIST:
    frag_dir = FRAGMENT_DIR_TEMPLATE.format(species=species)
    
    if not os.path.isdir(frag_dir):
        print(f"{species:<15} directory not found: {frag_dir}")
        continue
    
    files = sorted(glob.glob(os.path.join(frag_dir, FRAGMENT_PATTERN)))
    
    if files:
        SPECIES_FRAGMENTS[species] = files
        print(f"{species:<15} {len(files):>4} files   ({frag_dir})")
    else:
        print(f"{species:<15} 0 files matching {FRAGMENT_PATTERN} in {frag_dir}")

print("-" * 70)
total_files = sum(len(v) for v in SPECIES_FRAGMENTS.values())
print(f"{total_files} total fragment files across {len(SPECIES_FRAGMENTS)} species")

## Step 1: Convert Fragments → BigWig

This is a **one-time** step.  Each fragment file is converted to a bigWig file
using Tn5 cut-site coverage.  Existing bigWigs are automatically skipped.

BigWig files are much faster to query than raw fragment files because the signal
is pre-indexed in a binary R-tree format.

## Step 2: Quantify BigWigs Over Combined Peak Sets

Now that we have bigWig files, quantification is near-instant using
`pyBigWig.stats()` at the C level.

Each species gets **one matrix** (peaks × samples) containing both:
- **Unified peaks** (`unified_NNNNNN`) — comparable across genomes
- **Species-specific peaks** (`human_peak_NNNNNN` or `{species}_peak_NNNNNN`)

Saved as a single feather file per species.

In [ ]:
# === CONVERT ALL SPECIES' FRAGMENTS → BIGWIGS ===
# This is cached: existing .bw files are skipped automatically.

SPECIES_BIGWIGS = {}  # species -> list of .bw file paths

for species in SPECIES_LIST:
    if species not in SPECIES_FRAGMENTS:
        print(f"Skipping {species}: no fragment files")
        continue
    if species not in CHROM_SIZES:
        print(f"Skipping {species}: no chrom.sizes file")
        continue
    
    bw_dir = os.path.join(BIGWIG_OUTPUT_DIR, species)
    
    print(f"\n{'='*60}")
    print(f"{species}: {len(SPECIES_FRAGMENTS[species])} fragment files")
    print(f"{'='*60}")
    
    mapping = fragments_to_bigwigs(
        fragment_files=SPECIES_FRAGMENTS[species],
        chrom_sizes_file=CHROM_SIZES[species],
        output_dir=bw_dir,
        cut_sites=True,      # Tn5 cut-site mode (recommended for ATAC-seq)
        normalize=False,     # raw counts — better for DESeq2
        n_workers=8,
        verbose=True,
    )
    
    # Collect the bigWig paths (in same order as fragment files)
    bw_files = [mapping[f] for f in SPECIES_FRAGMENTS[species] if f in mapping]
    SPECIES_BIGWIGS[species] = sorted(bw_files)

print(f"\n{'='*60}")
print(f"BigWig files ready for {len(SPECIES_BIGWIGS)} species:")
for sp, bws in SPECIES_BIGWIGS.items():
    print(f"   {sp:<15} {len(bws)} bigWig files")

### Single bigWig test

Quick test with one file to verify everything works.

In [ ]:
# Quick test: quantify one bigWig file over peaks
test_species = next(
    (s for s in SPECIES_LIST if s in SPECIES_BIGWIGS and s in SPECIES_PEAK_FILES),
    None,
)

if test_species:
    test_bw = SPECIES_BIGWIGS[test_species][0]
    test_peaks = SPECIES_PEAK_FILES[test_species]
    
    print(f"Testing with {test_species}")
    print(f"   BigWig: {os.path.basename(test_bw)}")
    print(f"   Peaks:  {os.path.basename(test_peaks)}")
    
    result = quantify_bigwig(
        bigwig_file=test_bw,
        peak_file=test_peaks,
        stat="sum",       # "sum" | "mean" | "max" | "min" | "coverage"
        verbose=True,
    )
    
    print(f"\nResult shape: {result.shape}")
    print(f"   Non-zero peaks: {(result > 0).sum():,} / {len(result):,}")
    print(result.head(10))
else:
    print("No species has both bigWig and peak files. Check steps above.")

### Full Quantification Matrix

Quantify all bigWig files for one or all species.  The result is a **peaks × samples** matrix.

In [ ]:
# === QUANTIFY ONE SPECIES ===

SPECIES = "Human"  # Change to run a different species

if SPECIES in SPECIES_BIGWIGS and SPECIES in SPECIES_PEAK_FILES:
    bw_files  = SPECIES_BIGWIGS[SPECIES]
    peak_file = SPECIES_PEAK_FILES[SPECIES]
    
    print(f"Quantifying {SPECIES}: {len(bw_files)} bigWig files over {peak_file}")
    
    matrix = quantify_bigwig_matrix(
        bigwig_files=bw_files,
        peak_file=peak_file,
        stat="sum",               # "sum" | "mean" | "max" | "min" | "coverage"
        n_workers=8,
        output_file=os.path.join(QUANT_OUTPUT_DIR, f"quantification_{SPECIES}"),
        output_format="feather",
        verbose=True,
    )
    
    if matrix is not None:
        print(f"\nMatrix shape: {matrix.shape}")
        print(matrix.head())
else:
    print(f"Missing bigWigs or peaks for {SPECIES}. Check configuration above.")

In [ ]:
# === QUANTIFY ALL SPECIES ===
# Loops over every species that has both bigWigs and peak files.
# Each species gets one feather file containing unified + species-specific peaks.

for species in SPECIES_LIST:
    if species not in SPECIES_BIGWIGS or species not in SPECIES_PEAK_FILES:
        print(f"Skipping {species}: missing data")
        continue

    bw_files  = SPECIES_BIGWIGS[species]
    peak_file = SPECIES_PEAK_FILES[species]

    print(f"\n{'='*60}")
    print(f"{species}: {len(bw_files)} bigWig files → {os.path.basename(peak_file)}")
    print(f"{'='*60}")

    mat = quantify_bigwig_matrix(
        bigwig_files=bw_files,
        peak_file=peak_file,
        stat="sum",
        n_workers=8,
        output_file=os.path.join(QUANT_OUTPUT_DIR, f"quantification_{species}"),
        output_format="feather",
        verbose=True,
    )

    if mat is not None:
        idx = mat.index.astype(str)
        n_uni = idx.str.startswith("unified_").sum()
        print(f"   → {mat.shape[0]:,} peaks × {mat.shape[1]} samples "
              f"(unified: {n_uni:,}, specific: {mat.shape[0] - n_uni:,})")

## Load and Inspect Saved Matrix

In [ ]:
# Load and inspect a saved quantification matrix
SPECIES = "Human"  # change as needed

quant_path = os.path.join(QUANT_OUTPUT_DIR, f"quantification_{SPECIES}.feather")
if os.path.exists(quant_path):
    quant_df = load_matrix(quant_path)
    
    # Break down by peak type
    idx = quant_df.index.astype(str)
    n_unified  = idx.str.startswith("unified_").sum()
    n_specific = len(idx) - n_unified
    
    print(f"Matrix shape: {quant_df.shape}  ({quant_df.shape[0]:,} peaks × {quant_df.shape[1]} samples)")
    print(f"   Unified peaks:          {n_unified:,}")
    print(f"   Species-specific peaks:  {n_specific:,}")
    print(f"\nHead:")
    print(quant_df.head())
else:
    print(f"Not found: {quant_path}")
    print("Run the quantification cells above first.")

In [ ]:
# List all saved quantification files
print(f"Saved quantification files in {QUANT_OUTPUT_DIR}:")
print("-" * 60)

if os.path.isdir(QUANT_OUTPUT_DIR):
    for f in sorted(os.listdir(QUANT_OUTPUT_DIR)):
        fpath = os.path.join(QUANT_OUTPUT_DIR, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"   {f:<50} {size_mb:>8.1f} MB")
else:
    print("   (no output directory yet)")